In [51]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
import warnings
warnings.filterwarnings("ignore")

In [60]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)


In [64]:
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = {
    'drums': 'drums.wav',
    'vocals': 'vocals.wav',
    'bass': 'bass.wav',
    'other': 'other.wav'
}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0  # will update after finding rock.00006

In [65]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s: [] for s in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {s: [] for s in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)

    total_sounds = 0
    corrupted = 0
    size_below = 0
    size_above = 0
    CORRUPT_BYTES = 4 * 1024
    SMALL_BYTES   = 5.0491 * 1024**2
    LARGE_BYTES   = 5.0493 * 1024**2

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            continue

        song_dirs = sorted(os.listdir(genre_path))
        valid_songs = []

        for song in song_dirs:
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            stem_paths = {}
            complete = True
            for key in STEM_KEYS:
                fp = os.path.join(song_path, STEMS[key])
                if not os.path.exists(fp):
                    complete = False
                    break
                stem_paths[key] = fp

            if not complete:
                continue

            song_ok = True
            for key, fp in stem_paths.items():
                size = os.path.getsize(fp)
                total_sounds += 1

                if size < CORRUPT_BYTES:
                    corrupted += 1
                    song_ok = False

                if size < SMALL_BYTES:
                    size_below += 1

                if size > LARGE_BYTES:
                    size_above += 1

            if song_ok:
                valid_songs.append(stem_paths)

        rng.shuffle(valid_songs)
        n_val = int(len(valid_songs) * val_split)
        val_songs   = valid_songs[:n_val]
        train_songs = valid_songs[n_val:]

        def add_to_dict(target_dict, song_list):
            for sp in song_list:
                for key in STEM_KEYS:
                    target_dict[genre][key].append(sp[key])

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset,   val_songs)

    print(f"Total files scanned       : {total_sounds}")
    print(f"Corrupted (<4KB)          : {corrupted}")
    print(f"Files < 5.0491 MB         : {size_below}")
    print(f"Files > 5.0493 MB         : {size_above}")
    print(f"\nQ1 = corrupted + below    : {corrupted + size_below}")
    print(f"Q2 = |above - below|      : {abs(size_above - size_below)}")
    print(f"\nQ3 reggae drums (train)   : {0}")  # placeholder, shown after return
    print(f"Q3 country vocals (val)   : {0}")

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

# Q3
r = len(tr['reggae']['drums'])
c = len(val['country']['vocals'])
print(f"\nQ3 reggae drums train : {r}")
print(f"Q3 country vocals val : {c}")
print(f"Q3 answer             : {abs(r - c)}")

Total files scanned       : 4000
Corrupted (<4KB)          : 0
Files < 5.0491 MB         : 1256
Files > 5.0493 MB         : 184

Q1 = corrupted + below    : 1256
Q2 = |above - below|      : 1072

Q3 reggae drums (train)   : 0
Q3 country vocals (val)   : 0

Q3 reggae drums train : 83
Q3 country vocals val : 17
Q3 answer             : 66


In [66]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    total_files = sum(len(paths) for genre in dataset_dict for paths in dataset_dict[genre].values())
    print(f"Total files to scan: {total_files}")

    for genre in tqdm(dataset_dict):
        for stem_name, file_list in dataset_dict[genre].items():
            for file_path in file_list:
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                    total_duration = len(y) / sr
                    intervals = librosa.effects.split(y, top_db=top_db)

                    max_silence = 0.0
                    silence_type = []

                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type = ["full"]
                    else:
                        # START
                        start_sil = intervals[0][0] / sr
                        if start_sil > 0:
                            max_silence = max(max_silence, start_sil)
                            silence_type.append("start")

                        # END
                        end_sil = (len(y) - intervals[-1][1]) / sr
                        if end_sil > 0:
                            max_silence = max(max_silence, end_sil)
                            silence_type.append("end")

                        # MIDDLE — only if gap >= threshold
                        for i in range(len(intervals) - 1):
                            gap = (intervals[i+1][0] - intervals[i][1]) / sr
                            if gap >= threshold_sec:
                                max_silence = max(max_silence, gap)
                                if "middle" not in silence_type:
                                    silence_type.append("middle")

                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })

                except Exception as e:
                    print(f"Error {file_path}: {e}")

    return pd.DataFrame(records)

df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

print(f"\nQ4  - Total silence >= 5s     : {len(df_silence)}")

vocals_df  = df_silence[df_silence['Stem'] == 'vocals']
print(f"Q5  - Vocals silence >= 5s    : {len(vocals_df)}")
print(f"Q6  - Avg silence in Vocals   : {vocals_df['Max_Silence_Sec'].mean():.2f}")

jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
print(f"Q7  - Jazz drums >= 5s        : {len(jazz_drums)}")
print(f"Q8  - Jazz drums only middle  : {len(jazz_drums[jazz_drums['Silence_Location'] == 'middle'])}")
print(f"Q9  - Jazz drums >= 10s       : {len(jazz_drums[jazz_drums['Max_Silence_Sec'] >= 10])}")

print("\n--- Pivot Table ---")
pivot = df_silence.pivot_table(index='Genre', columns='Stem', values='File_Path', aggfunc='count', fill_value=0)
print(pivot)

Total files to scan: 3320


100%|██████████| 10/10 [05:19<00:00, 31.91s/it]


Q4  - Total silence >= 5s     : 688
Q5  - Vocals silence >= 5s    : 312
Q6  - Avg silence in Vocals   : 12.54
Q7  - Jazz drums >= 5s        : 22
Q8  - Jazz drums only middle  : 0
Q9  - Jazz drums >= 10s       : 8

--- Pivot Table ---
Stem       bass  drums  other  vocals
Genre                                
blues        17     29      4      44
classical    68     58      5      69
country      12     14      1      16
disco         8      2      2      17
hiphop       22      2     21       6
jazz         24     22      1      73
metal         7      2      1      41
pop          11      6      2       5
reggae        5      5      7      15
rock         12      5      1      26


In [67]:
stems_audio = []
try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        print(f"Loading [{key}]: {file_path}")
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)
        print(f"  shape: {y.shape}")
    print("\nAudio loaded successfully.")
except Exception as e:
    print(f"ERROR: {e}")

Loading [drums]: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00082/drums.wav
  shape: (110250,)
Loading [vocals]: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00082/vocals.wav
  shape: (110250,)
Loading [bass]: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00082/bass.wav
  shape: (110250,)
Loading [other]: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00082/other.wav
  shape: (110250,)

Audio loaded successfully.


In [68]:
stems_stack = np.array(stems_audio)
print(f"Stems stack shape : {stems_stack.shape}")

mix_raw = np.sum(stems_stack, axis=0)
print(f"Q10 - Mix length  : {len(mix_raw)}")

rms_val = np.sqrt(np.mean(mix_raw ** 2))
print(f"Q11 - RMS         : {rms_val:.2f}")

max_val = np.max(np.abs(mix_raw))
print(f"Peak before norm  : {max_val:.2f}")

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

print(f"Q12 - Max normalized : {np.max(np.abs(mix_norm)):.2f}")
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
print("Normalization assertion passed")

Stems stack shape : (4, 110250)
Q10 - Mix length  : 110250
Q11 - RMS         : 0.09
Peak before norm  : 0.56
Q12 - Max normalized : 1.00
Normalization assertion passed
